In [3]:
# Stable Diffusion 示例代码
# 本示例简化了 Stable Diffusion 的整体流程，包含 text encoder、U-Net 生成器 和 解码器（VAE Decoder）模块
# 注意：实际部署需要大量 GPU 资源，此处用于教学展示

import torch  # 导入PyTorch库
import torch.nn as nn  # 导入神经网络模块
import torch.nn.functional as F  # 导入函数式接口
from transformers import CLIPTextModel, CLIPTokenizer  # 导入CLIP模型和分词器

# 1. 文本编码器（CLIP Text Encoder）
class TextEncoder:  # 定义文本编码器类
    def __init__(self, device="cuda"):  # 初始化方法，默认使用CUDA设备
        self.tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")  # 加载CLIP分词器
        self.text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)  # 加载CLIP文本编码器并移至指定设备
        self.device = device  # 保存设备信息

    def encode(self, prompt):  # 编码方法，将文本提示转换为嵌入向量
        tokens = self.tokenizer(prompt, return_tensors="pt", padding=True).to(self.device)  # 将提示文本转换为token并移至设备
        with torch.no_grad():  # 不计算梯度
            text_embeddings = self.text_encoder(**tokens).last_hidden_state  # [B, 77, 768] 获取文本嵌入
        return text_embeddings  # 返回文本嵌入

# 简单的交叉注意力模块
class CrossAttention(nn.Module):  # 定义交叉注意力模块
    def __init__(self, latent_dim, context_dim):  # 初始化方法，接收潜在维度和上下文维度
        super().__init__()  # 调用父类初始化
        self.query = nn.Linear(latent_dim, latent_dim)  # 查询线性层
        self.key = nn.Linear(context_dim, latent_dim)  # 键线性层
        self.value = nn.Linear(context_dim, latent_dim)  # 值线性层
        self.out = nn.Linear(latent_dim, latent_dim)  # 输出线性层

    def forward(self, x, context):  # 前向传播方法
        # x: [B, C, H, W] -> [B, HW, C]
        B, C, H, W = x.shape  # 获取输入张量的形状
        x_flat = x.view(B, C, H * W).transpose(1, 2)  # [B, HW, C] 将特征图展平并转置
        q = self.query(x_flat)          # [B, HW, C] 计算查询向量
        k = self.key(context)           # [B, T, C] 计算键向量
        v = self.value(context)         # [B, T, C] 计算值向量

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (C ** 0.5)  # [B, HW, T] 计算注意力分数
        attn_weights = F.softmax(attn_scores, dim=-1)                   # [B, HW, T] 计算注意力权重
        attended = torch.matmul(attn_weights, v)                        # [B, HW, C] 应用注意力

        out = self.out(attended).transpose(1, 2).view(B, C, H, W)       # [B, C, H, W] 重塑回原始形状
        return out  # 返回注意力输出

# 2. U-Net 生成器 (增加交叉注意力)
class UNet(nn.Module):  # 定义U-Net模型
    def __init__(self):  # 初始化方法
        super().__init__()  # 调用父类初始化
        self.down = nn.Sequential(  # 下采样路径
            nn.Conv2d(4, 64, 3, padding=1),  # 第一个卷积层
            nn.ReLU(),  # ReLU激活函数
            nn.Conv2d(64, 128, 3, padding=1),  # 第二个卷积层
            nn.ReLU()  # ReLU激活函数
        )
        # 修正：确保 latent_dim 和 context_dim 匹配
        self.cross_attn = CrossAttention(latent_dim=128, context_dim=768)  # 交叉注意力层
        self.mid = nn.Sequential(  # 中间层
            nn.Conv2d(128, 128, 3, padding=1),  # 卷积层
            nn.ReLU()  # ReLU激活函数
        )
        self.up = nn.Sequential(  # 上采样路径
            nn.ConvTranspose2d(128, 64, 3, padding=1),  # 第一个转置卷积层
            nn.ReLU(),  # ReLU激活函数
            nn.ConvTranspose2d(64, 4, 3, padding=1)  # 第二个转置卷积层
        )

    def forward(self, x, text_emb):  # 前向传播方法
        x = self.down(x)                      # [B, 128, H, W] 下采样
        x = self.cross_attn(x, text_emb)      # 加入文本条件
        x = self.mid(x)  # 中间处理，中间层功能见课件
        x = self.up(x)  # 上采样
        return x  # 返回结果

# 3. 解码器 VAE (简化版)
class Decoder(nn.Module):  # 定义解码器类
    def __init__(self):  # 初始化方法
        super().__init__()  # 调用父类初始化
        self.net = nn.Sequential(  # 解码网络
            nn.Conv2d(4, 64, 3, padding=1),  # 第一个卷积层
            nn.ReLU(),  # ReLU激活函数
            nn.Conv2d(64, 3, 3, padding=1),  # 第二个卷积层
            nn.Tanh()  # Tanh激活函数，输出范围[-1,1]
        )

    def forward(self, x):  # 前向传播方法
        return self.net(x)  # 返回解码结果

# 4. 整体流程封装
class StableDiffusionDemo:  # 定义Stable Diffusion演示类
    def __init__(self, device="cuda"):  # 初始化方法
        self.device = device  # 保存设备信息
        self.text_encoder = TextEncoder(device)  # 初始化文本编码器
        self.unet = UNet().to(device)  # 初始化U-Net并移至设备
        self.decoder = Decoder().to(device)  # 初始化解码器并移至设备

    def generate(self, prompt):  # 图像生成方法
        # 获取文本嵌入
        text_embeddings = self.text_encoder.encode(prompt)  # 编码提示文本
        
        # 确保文本嵌入的维度正确
        batch_size = text_embeddings.shape[0]  # 获取批次大小
        seq_len = text_embeddings.shape[1]  # 获取序列长度
        hidden_dim = text_embeddings.shape[2]  # 获取隐藏维度
        
        # 如果需要，可以调整文本嵌入的维度以匹配交叉注意力的需求
        # 确保文本嵌入的维度是 [B, seq_len, 768]
        if hidden_dim != 768:  # 如果隐藏维度不是768
            # 创建一个投影层来调整维度
            projection = nn.Linear(hidden_dim, 768).to(self.device)  # 创建投影层
            text_embeddings = projection(text_embeddings)  # 应用投影
        
        latent = torch.randn(batch_size, 4, 64, 64).to(self.device)  # 初始 latent，随机噪声
        for _ in range(10):  # 简化版的 DDIM 迭代过程，进行10次迭代
            noise_pred = self.unet(latent, text_embeddings)  # 预测噪声
            latent = latent - noise_pred * 0.1  # 去噪步骤
        image = self.decoder(latent)  # 解码生成图像
        return image  # 返回生成的图像

# 示例运行
device = "cuda" if torch.cuda.is_available() else "cpu"  # 检查是否有GPU可用
generator = StableDiffusionDemo(device)  # 创建生成器实例
output = generator.generate("a cat riding a bike")  # 生成"猫骑自行车"的图像
print("图像生成完成，输出张量 shape:", output.shape)  # 输出: [1, 3, 64, 64] 打印输出图像的形状


图像生成完成，输出张量 shape: torch.Size([1, 3, 64, 64])


In [ ]:
# 导入必要的库
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import os

def generate_image_from_text(prompt: str, output_path: str = "generated_image.png", model_id: str = "runwayml/stable-diffusion-v1-5", device: str = "cuda", print_model_structure: bool = False):
    """
    使用 Stable Diffusion 根据文本提示生成图像。

    参数:
    prompt (str): 用于生成图像的文本提示。
    output_path (str): 生成图像的保存路径。
    model_id (str): Hugging Face Hub 上的预训练模型 ID。
                     一些流行的模型包括:
                     - "runwayml/stable-diffusion-v1-5" (通用)
                     - "stabilityai/stable-diffusion-2-1" (通用)
                     - "CompVis/stable-diffusion-v1-4" (较早版本)
                     - 还有许多针对特定风格微调的模型，例如 "dreamlike-art/dreamlike-photoreal-2.0"
    device (str): 用于推理的设备 ("cuda" 表示 GPU, "cpu" 表示 CPU)。
                  如果 CUDA 不可用，会自动切换到 CPU。
    print_model_structure (bool): 是否打印模型的主要组件结构。
    """
    # print(f"开始生成图像，提示: '{prompt}'")
    # print(f"使用模型: {model_id}")

    # 检查 CUDA 是否可用，否则使用 CPU
    if device == "cuda" and not torch.cuda.is_available():
        # print("CUDA 不可用，切换到 CPU。生成过程可能会很慢。")
        device = "cpu"

    # print(f"使用设备: {device}")

    try:
        # 加载预训练的 Stable Diffusion pipeline
        # torch_dtype=torch.float16 在 GPU 上可以加速并减少显存占用
        # 如果在 CPU 上运行或 GPU 不支持 float16，可以移除 torch_dtype 参数或使用 torch.float32
        pipe_args = {}
        if device == "cuda":
            pipe_args["torch_dtype"] = torch.float16
            # 如果显存较小，可以启用 VAE 分块处理以减少显存峰值
            # pipe.enable_vae_slicing()
            # 也可以启用模型 offloading，进一步减少显存占用，但会牺牲一些速度
            # pipe.enable_model_cpu_offload()

        print("正在加载模型... 这可能需要一些时间，特别是第一次运行时会自动下载模型。")
        pipe = StableDiffusionPipeline.from_pretrained(model_id, **pipe_args)
        pipe = pipe.to(device)
        print("模型加载完成。")

        # 如果设置了 print_model_structure 为 True，则打印模型结构
        if print_model_structure:
            # print("\n--- 开始打印 Stable Diffusion 模型结构 ---")

            if hasattr(pipe, 'vae') and pipe.vae is not None:
                # print("\n1. VAE (Variational Autoencoder) 结构:")
                # print(pipe.vae)
                pass
            else:
                # print("\n1. VAE: 未加载或不可用")
                pass

            if hasattr(pipe, 'text_encoder') and pipe.text_encoder is not None:
                # print("\n2. Text Encoder 结构:")
                # print(pipe.text_encoder)
                pass
            else:
                # print("\n2. Text Encoder: 未加载或不可用")
                pass

            if hasattr(pipe, 'tokenizer') and pipe.tokenizer is not None:
                # print("\n3. Tokenizer 信息:")
                # print(pipe.tokenizer) # Tokenizer 不是 nn.Module, 打印其类和配置
                pass
            else:
                # print("\n3. Tokenizer: 未加载或不可用")
                pass

            if hasattr(pipe, 'unet') and pipe.unet is not None:
                # print("\n4. U-Net 结构:")
                # print(pipe.unet)
                pass
            else:
                # print("\n4. U-Net: 未加载或不可用")
                pass

            if hasattr(pipe, 'scheduler') and pipe.scheduler is not None:
                # print("\n5. Scheduler 信息:")
                # print(pipe.scheduler) # Scheduler 不是 nn.Module, 打印其类和配置
                pass
            else:
                # print("\n5. Scheduler: 未加载或不可用")
                pass

            if hasattr(pipe, 'safety_checker') and pipe.safety_checker is not None:
                # print("\n6. Safety Checker 结构:")
                # print(pipe.safety_checker)
                pass
            else:
                # print("\n6. Safety Checker: 未加载或不可用")
                pass

            if hasattr(pipe, 'feature_extractor') and pipe.feature_extractor is not None:
                # print("\n7. Feature Extractor (通常用于 Safety Checker) 信息:")
                # print(pipe.feature_extractor) # 通常是 PreTrainedFeatureExtractor, 打印其类和配置
                pass
            else:
                # print("\n7. Feature Extractor: 未加载或不可用")
                pass

            # print("\n--- 模型结构打印完成 ---\n")

        # 生成图像
        # num_inference_steps 控制去噪步数，通常 50 步效果不错，增加步数可以提升细节但也会增加时间
        # guidance_scale 控制提示与图像的符合程度，值越高越符合提示，但可能牺牲图像质量或多样性，通常 7-8.5 效果较好
        print("正在生成图像...")
        with torch.no_grad(): # 在推理时禁用梯度计算以节省内存和加速
            # 如果需要设置随机种子: generator = torch.Generator(device=device).manual_seed(42)
            # image = pipe(prompt, num_inference_steps=50, guidance_scale=7.5, generator=generator).images[0]
            image = pipe(prompt, num_inference_steps=50, guidance_scale=7.5).images[0]

        print("图像生成完成。")

        # 保存图像
        image.save(output_path)
        # print(f"图像已保存到: {output_path}")

        # （可选）显示图像 (如果环境支持)
        # try:
        #     image.show()
        # except Exception as e:
        #     print(f"无法显示图像: {e} (可能在无头服务器上运行)")

    except Exception as e:
        print(f"生成图像或打印模型结构时发生错误: {e}")
        if "out of memory" in str(e).lower():
            print("显存不足 (Out of Memory)。请尝试以下方法：")
            print("1. 如果您有多个GPU，请确保 CUDA_VISIBLE_DEVICES 环境变量设置正确。")
            print("2. 尝试使用更小的模型（如果可用）。")
            print("3. 减小图像生成尺寸（如果 pipeline 支持 width/height 参数）。")
            print("4. 如果在 GPU 上运行，确保已使用 torch_dtype=torch.float16。")
            print("5. 尝试启用 'pipe.enable_vae_slicing()' 或 'pipe.enable_model_cpu_offload()'。")
            print("6. 关闭其他占用显存的程序。")
            print("7. 如果使用的是 CPU，确保有足够的 RAM。")
        elif "requires a GPU" in str(e).lower() and device == "cpu":
             print("模型可能配置为仅在 GPU 上运行。请检查模型文档或尝试其他模型。")


if __name__ == "__main__":
    # --- 用户配置 ---
    # 你想要生成的图像的描述
    # text_prompt = "一只穿着宇航服的猫在月球上弹吉他，数字艺术风格"
    # text_prompt = "A photo of an astronaut riding a horse on the moon" # 英文提示通常效果更好
    text_prompt = "A photo of a dog diving on the back" # 英文提示通常效果更好

    # 生成图像的保存文件名
    output_image_path = "dog_dive.png"

    # 选择模型 (更多模型请访问 https://huggingface.co/models?pipeline_tag=text-to-image&sort=downloads)
    # model_name = "runwayml/stable-diffusion-v1-5"
    model_name = "stabilityai/stable-diffusion-2-1-base" # 512x512 基础模型
    # model_name = "stabilityai/stable-diffusion-xl-base-1.0" # SDXL 模型，质量更高，但需要更多资源
                                                            # 使用 SDXL 时，pipeline 可能需要换成 StableDiffusionXLPipeline

    # 选择设备 ("cuda" 或 "cpu")
    # 如果您有 NVIDIA GPU 并且已正确安装 CUDA 和 PyTorch GPU 版本，请使用 "cuda"
    # 否则，使用 "cpu" (会非常慢)
    processing_device = "cuda"
    # processing_device = "cpu"

    # 是否打印模型结构
    should_print_structure = True

    # --- 执行 ---
    generate_image_from_text(
        prompt=text_prompt,
        output_path=output_image_path,
        model_id=model_name,
        device=processing_device,
        print_model_structure=should_print_structure # 传递参数以打印结构
    )

    # 如果只想打印模型结构而不生成图像，可以创建一个单独的函数或调整逻辑
    # 例如:
    # if should_print_structure:
    #     # 只加载模型并打印结构，不进行图像生成
    #     try:
    #         pipe = StableDiffusionPipeline.from_pretrained(model_name, torch_dtype=torch.float16 if processing_device == "cuda" else torch.float32)
    #         pipe = pipe.to(processing_device)
    #         # ... (在此处复制打印结构的代码块) ...
    #     except Exception as e:
    #         print(f"加载或打印模型结构时出错: {e}")
    # else:
    #     # 正常生成图像
    #     generate_image_from_text(
    #         prompt=text_prompt,
    #         output_path=output_image_path,
    #         model_id=model_name,
    #         device=processing_device,
    #         print_model_structure=False
    #     )


    # --- 更多示例提示 ---
    # generate_image_from_text("A futuristic cityscape at sunset, with flying cars and neon lights, concept art", "futuristic_city.png", device=processing_device)
    # generate_image_from_text("A hyperrealistic portrait of an old fisherman with a weathered face, detailed skin texture", "fisherman_portrait.png", device=processing_device)
    # generate_image_from_text("An oil painting of a mystical forest with glowing mushrooms and ancient trees, fantasy art", "mystical_forest.png", device=processing_device)